# Multi-Object Tracking and Data Association

Detection answers *what is in this frame*. Tracking answers *which of these is the same object
as last frame*. This lab is about the second question — the **data association** step that turns
a pile of per-frame boxes into object identities that persist over time.

You will run all six Ultralytics trackers over the same video with the same detector, compare
what changes, and — importantly — learn what those differences do and do **not** let you claim.

**Inputs**

- A fine-tuned detector checkpoint (`best.pt`) from any downstream lab in this series
- A football video, e.g. `/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/video.mp4`

**Learning goals**

- Separate the detector from the tracker, and know which one you actually changed
- Read a tracker configuration and predict what raising `track_buffer` will do
- Compare six trackers on identical detections and interpret the result honestly
- Explain why MOTA, IDF1, and HOTA cannot be computed from a plain video
- Understand exactly where SSL pretraining does and does not affect tracking

**Prerequisite:** `ssl-detection-lab` **0.9.0+**. Earlier versions have no tracker catalog.

## The tracking pipeline

```
frame ──▶ detector ──▶ boxes + scores ──▶ association ──▶ track IDs
                            │                   │
                     SSL pretraining      tracker config
                     changes THIS         changes THIS
```

A tracker is **not trained** by this package. It is an algorithm with hyperparameters that
consumes whatever your detector emits. This matters for how you write up results:

- A better backbone improves tracking **only** by improving the detections fed to the tracker.
- Swapping BoT-SORT for ByteTrack changes association only. The detections are byte-identical.

Keeping those two straight is the whole point of the lab.

## Notebook roadmap

1. Configure the runtime and install the library
2. Inspect the tracker catalog
3. Locate the detector weights and the video
4. First tracked run with BoT-SORT
5. Read the recorded tracker settings
6. Track lifetimes and fragmentation
7. Compare all six trackers
8. Interpret the comparison honestly
9. Tune one tracker with a custom config
10. Where SSL fits — and where it does not
11. Export the results

## 1. Configure the runtime

On Kaggle: **Session options → Accelerator → GPU T4 x2**, **Internet on**, and attach the
football dataset via **Add Input**. Tracking is inference-only, so a single GPU — or even CPU
with a small `MAX_FRAMES` — is enough for this lab.

In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps "git+https://github.com/rifat963/ssl-detection-lab.git@main"

In [ ]:
import json
import shutil
from pathlib import Path
from importlib.metadata import version as installed_version

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Markdown, display
from packaging.version import Version

from ssldet import (
    TRACKERS,
    VideoAnalysisConfig,
    analyze_video,
    available_tracker_names,
    capabilities,
    resolve_tracker,
)

INSTALLED = installed_version("ssl-detection-lab")
assert Version(INSTALLED) >= Version("0.9.0"), (
    f"Installed ssl-detection-lab {INSTALLED}; this lab needs 0.9.0 or newer for the tracker "
    "catalog, fail-fast tracker validation, and recorded tracker settings."
)

pd.set_option("display.max_colwidth", 60)
print(f"ssl-detection-lab {INSTALLED}")
print("trackers:", ", ".join(available_tracker_names()))

## 2. The tracker catalog

The catalog is plain data — it renders without loading PyTorch or Ultralytics, so you can read it
before committing any GPU time.

In [ ]:
trackers = pd.DataFrame(capabilities()["trackers"])
display(
    trackers[
        ["name", "association", "appearance_reid", "motion_compensation", "pick_it_when"]
    ].rename(
        columns={
            "appearance_reid": "ReID",
            "motion_compensation": "motion comp.",
            "pick_it_when": "pick it when",
        }
    )
)

Two columns decide most of the cost/quality trade-off:

- **ReID** — the tracker compares *appearance* embeddings, not just box overlap. It survives
  occlusion and crossing paths far better, at the price of an extra model per frame.
  It is **off by default** everywhere (`with_reid: False`); you must opt in.
- **motion comp.** — global motion compensation estimates camera movement between frames.
  Essential for handheld or panning footage, wasted compute on a locked-off camera.

The equivalent from a terminal is `ssldet trackers` (add `--json` for machine-readable output).

In [ ]:
for tracker in TRACKERS:
    print(f"{tracker.name:11} {tracker.paper_url}")

## 3. Locate the detector weights and the video

Tracking needs a **fine-tuned detector** checkpoint. `best_ssl.pt` is SSL training state, not a
detector — passing it here will not work. Use the `best.pt` produced by any downstream lab, or
fall back to COCO-pretrained `yolo26n.pt` to exercise the pipeline.

In [ ]:
WEIGHTS_CANDIDATES = [
    "/kaggle/working/simclr_yolo26_football/finetune/weights/best.pt",
    "/kaggle/working/byol_yolo26_football/finetune/weights/best.pt",
    "runs/finetuned/weights/best.pt",
    "yolo26n.pt",  # COCO fallback: pipeline works, classes are COCO not football
]
VIDEO_CANDIDATES = [
    "/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/video.mp4",
    "/kaggle/input/football-player-detection-yolov8/video.mp4",
    "match.mp4",
]


def first_existing(candidates, label):
    for candidate in candidates:
        if Path(candidate).exists():
            return candidate
    # yolo26n.pt is downloaded on demand by Ultralytics, so it is a valid last resort.
    if label == "weights":
        return candidates[-1]
    raise FileNotFoundError(
        f"No {label} found. Tried:\n  " + "\n  ".join(candidates) +
        "\nEdit the candidate list above, or set the path directly."
    )


WEIGHTS = first_existing(WEIGHTS_CANDIDATES, "weights")
VIDEO = first_existing(VIDEO_CANDIDATES, "video")
MODEL_NAME = "yolo26n"
MAX_FRAMES = 300      # bounded so the six-tracker sweep stays quick; raise for real work
OUTPUT_ROOT = Path("/kaggle/working/tracker_lab")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("weights:", WEIGHTS)
print("video:  ", VIDEO)
if WEIGHTS.endswith("yolo26n.pt"):
    print("\nNOTE: using the COCO fallback. Classes will be COCO ('person', 'sports ball'),")
    print("not your football classes. The tracking lesson is identical.")

### Validation happens before any GPU time

A mistyped tracker name is rejected by `.validate()` — before weights load and before a single
frame is decoded. This is worth seeing once, because the failure mode it replaces was a crash
several minutes into a long video.

In [ ]:
try:
    VideoAnalysisConfig(
        video_source=VIDEO,
        model_name=MODEL_NAME,
        weights_file=WEIGHTS,
        tracker="bytrack.yaml",   # deliberate typo
    ).validate()
except ValueError as error:
    print("Rejected at configuration time:\n")
    print(error)

## 4. First tracked run with BoT-SORT

BoT-SORT is the default: IoU association, optional ReID, and global motion compensation. It is
the sensible starting point for broadcast-style footage where the camera moves.

In [ ]:
botsort_result = analyze_video(
    VideoAnalysisConfig(
        video_source=VIDEO,
        model_name=MODEL_NAME,
        weights_file=WEIGHTS,
        output_dir=OUTPUT_ROOT / "botsort",
        tracker="botsort.yaml",
        max_frames=MAX_FRAMES,
        save_annotated=True,
    ).validate()
)

report = json.loads(Path(botsort_result.report_json).read_text(encoding="utf-8"))
metrics = report["video_metrics"]
print(f"frames processed : {metrics['frames_processed']}")
print(f"total detections : {metrics['total_detections']}")
print(f"unique tracks    : {metrics['tracking']['unique_tracks']}")
print(f"processing FPS   : {metrics['processing_fps']:.2f}")

## 5. The report records the *resolved* tracker settings

This is the part that makes a tracked run reproducible. Recording the string `"botsort.yaml"`
would be nearly useless — those defaults change between Ultralytics releases, and a custom config
would be invisible to whoever reads your report six months from now. The report stores the
settings that were actually in force.

In [ ]:
tracking = report["video_metrics"]["tracking"]
print(f"tracker      : {tracking['tracker']}")
print(f"tracker_type : {tracking['tracker_type']}")
print(f"built-in     : {tracking['is_builtin']}")
print(f"resolved     : {tracking['resolved']}\n")

display(
    pd.DataFrame(
        sorted(tracking["settings"].items()), columns=["setting", "value"]
    ).set_index("setting")
)

The settings worth understanding before you tune anything:

| Setting | Effect of raising it |
|---|---|
| `track_buffer` | Lost tracks survive longer → fewer fragments, more risk of reusing an ID on the wrong object |
| `match_thresh` | Association gets stricter → fewer wrong matches, more fragments |
| `new_track_thresh` | Harder to spawn a track → fewer spurious tracks, more missed objects |
| `track_high_thresh` | Only confident detections drive the first pass → cleaner but sparser tracks |
| `with_reid` | Appearance matching on → far better through occlusion, noticeably slower |

## 6. Track lifetimes and fragmentation

`detections.csv` has one row per detection with its `track_id`. Track *lifetime* — how many
sampled frames an ID survives — is the most informative thing you can measure without labels.

A healthy tracker on this kind of footage produces a few long tracks. A tracker that is
fragmenting produces many short ones.

In [ ]:
detections = pd.read_csv(botsort_result.detections_csv)
tracked = detections[detections["track_id"].notna()]

lifetimes = tracked.groupby("track_id")["frame"].agg(["count", "min", "max"])
lifetimes["span"] = lifetimes["max"] - lifetimes["min"] + 1
lifetimes["continuity"] = lifetimes["count"] / lifetimes["span"]

print(f"tracks              : {len(lifetimes)}")
print(f"median lifetime     : {lifetimes['count'].median():.0f} sampled frames")
print(f"tracks lasting 1-2  : {(lifetimes['count'] <= 2).sum()}")
print(f"median continuity   : {lifetimes['continuity'].median():.2f}  (1.00 = never skipped)")

figure, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(lifetimes["count"], bins=30, color="#2a9d8f", edgecolor="white")
axes[0].set_xlabel("track lifetime (sampled frames)")
axes[0].set_ylabel("number of tracks")
axes[0].set_title("Most tracks should not be short")

per_frame = tracked.groupby("frame")["track_id"].nunique()
axes[1].plot(per_frame.index, per_frame.values, color="#e76f51", linewidth=1.2)
axes[1].set_xlabel("frame")
axes[1].set_ylabel("simultaneous tracks")
axes[1].set_title("Active tracks over time")
figure.tight_layout()
plt.show()

> **A short track is ambiguous.** It can mean the tracker dropped an object it should have kept,
> *or* that an object genuinely entered and left the frame. Without ground-truth identities you
> cannot tell these apart — which is precisely why the next sections refuse to declare a winner.

## 7. Compare all six trackers

Same detector, same weights, same frames, same thresholds. **Only the association algorithm
changes.** Any difference below is attributable to the tracker and nothing else — this is the
controlled comparison the lab is built around.

In [ ]:
comparison = []
artifacts = {}

for name in available_tracker_names():
    outcome = analyze_video(
        VideoAnalysisConfig(
            video_source=VIDEO,
            model_name=MODEL_NAME,
            weights_file=WEIGHTS,
            output_dir=OUTPUT_ROOT / name,
            tracker=f"{name}.yaml",
            max_frames=MAX_FRAMES,
            save_annotated=False,   # skip encoding; we only want the numbers
        ).validate()
    )
    artifacts[name] = outcome
    payload = json.loads(Path(outcome.report_json).read_text(encoding="utf-8"))
    video_metrics = payload["video_metrics"]
    rows = pd.read_csv(outcome.detections_csv)
    tracked_rows = rows[rows["track_id"].notna()]
    spans = tracked_rows.groupby("track_id")["frame"].count() if len(tracked_rows) else pd.Series(dtype=float)

    comparison.append(
        {
            "tracker": name,
            "detections": video_metrics["total_detections"],
            "unique_tracks": video_metrics["tracking"]["unique_tracks"],
            "median_lifetime": float(spans.median()) if len(spans) else 0.0,
            "short_tracks(<=2)": int((spans <= 2).sum()) if len(spans) else 0,
            "processing_fps": round(video_metrics["processing_fps"], 2),
        }
    )
    print(f"{name:11} done")

results = pd.DataFrame(comparison).set_index("tracker")
display(results)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = "#264653"

results["unique_tracks"].plot.bar(ax=axes[0], color=palette)
axes[0].set_title("Unique track IDs")
axes[0].set_ylabel("count")

results["median_lifetime"].plot.bar(ax=axes[1], color="#2a9d8f")
axes[1].set_title("Median track lifetime")
axes[1].set_ylabel("sampled frames")

results["processing_fps"].plot.bar(ax=axes[2], color="#e9c46a")
axes[2].set_title("End-to-end throughput")
axes[2].set_ylabel("FPS")

for axis in axes:
    axis.tick_params(axis="x", rotation=45)
figure.tight_layout()
plt.show()

In [ ]:
detection_counts = results["detections"].unique()
print("distinct detection totals across trackers:", detection_counts)
if len(detection_counts) == 1:
    print("\nAs expected: identical detections. Every difference above is association only.")
else:
    print("\nDetection totals differ. Some trackers filter detections by their own")
    print("track_high_thresh before association, so the surviving count can vary.")

## 8. Interpreting the comparison honestly

This is the section to read twice before writing anything up.

**What the table above supports**

- A throughput ranking on this hardware, this video, and this frame budget.
- A statement that tracker A produced more/fewer IDs and longer/shorter tracks than tracker B
  *on identical detections*.

**What it does not support**

- "Tracker A is more accurate." You have no ground-truth identities, so you cannot tell a
  correct new track from an ID switch.
- "Fewer unique tracks is better." Fewer IDs can mean better continuity **or** two players
  merged into one ID. Both look identical in this table.
- "Longer tracks are better." A single ID drifting across three different players is very long
  and completely wrong.

**The metric boundary, restated**

| Available here | Needs ground-truth identities |
|---|---|
| unique tracks, track lifetime, continuity | MOTA, MOTP |
| detections per frame, confidence stats | IDF1, HOTA |
| latency, throughput, frame coverage | ID switches, fragmentation rate |

`video_analysis.json` states this in its `metric_boundary` field, and `outcome.md` repeats it.
To make an accuracy claim you need a labelled MOT sequence — MOT17, MOT20, DanceTrack, or your
own annotated clip — evaluated with a MOT metrics library such as `TrackEval`.

In [ ]:
boundary = report["metric_boundary"]
print("Measurable without labels:")
for item in boundary["available_without_labels"]:
    print(f"  - {item}")
print("\nRequires ground truth:")
for item in boundary["requires_ground_truth"]:
    print(f"  - {item}")

## 9. Tune one tracker with a custom config

The tracker YAML files ship with Ultralytics under **AGPL-3.0**, so this MIT-licensed package
does not vendor copies of them. Copy one out of your install, edit it, and pass the local path —
`.validate()` accepts any existing YAML file.

Here we sweep `track_buffer`, which controls how many frames a lost track stays revivable.

In [ ]:
from ultralytics.utils.checks import check_yaml

base_config = Path(check_yaml("botsort.yaml"))
print("resolved base config:", base_config)

sweep = []
for buffer_frames in (5, 30, 120):
    custom_path = OUTPUT_ROOT / f"botsort_buffer{buffer_frames}.yaml"
    settings = yaml.safe_load(base_config.read_text(encoding="utf-8"))
    settings["track_buffer"] = buffer_frames
    custom_path.write_text(yaml.safe_dump(settings, sort_keys=False), encoding="utf-8")

    outcome = analyze_video(
        VideoAnalysisConfig(
            video_source=VIDEO,
            model_name=MODEL_NAME,
            weights_file=WEIGHTS,
            output_dir=OUTPUT_ROOT / f"buffer{buffer_frames}",
            tracker=str(custom_path),
            max_frames=MAX_FRAMES,
            save_annotated=False,
        ).validate()
    )
    payload = json.loads(Path(outcome.report_json).read_text(encoding="utf-8"))
    recorded = payload["video_metrics"]["tracking"]
    sweep.append(
        {
            "track_buffer": buffer_frames,
            "unique_tracks": recorded["unique_tracks"],
            "is_builtin": recorded["is_builtin"],
            "recorded_buffer": recorded["settings"]["track_buffer"],
        }
    )

display(pd.DataFrame(sweep).set_index("track_buffer"))

Two things to notice:

1. **`unique_tracks` falls as `track_buffer` rises.** Longer memory lets a track survive an
   occlusion instead of dying and respawning with a fresh ID. Whether that is *correct* is,
   again, unknowable without labels — a long buffer also lets an ID migrate to the wrong player.
2. **`is_builtin` is `False` and `recorded_buffer` matches what you set.** Your custom config
   was captured in the report. Anyone reading it later can reconstruct the exact run.

## 10. Where SSL fits — and where it does not

This lab sits in an SSL course, so be precise about the causal chain:

```
unlabelled images ──▶ SSL pretraining ──▶ better backbone
                                              │
                                              ▼
                                     better DETECTIONS
                                              │
                                              ▼
                              tracker (unchanged) ──▶ better tracks
```

**SSL pretraining never touches the tracker.** BoT-SORT's `match_thresh` is the same number
whether your backbone was randomly initialized, COCO-pretrained, or SimCLR-pretrained.

So if you want to claim SSL improved tracking, the honest experiment is:

1. Hold the tracker and **all** its settings fixed.
2. Vary only the detector initialization (random / COCO / SSL / COCO+SSL).
3. Report labelled detection metrics (mAP50-95) from the evaluator — those you *can* measure.
4. If you additionally claim tracking improved, you need a labelled MOT sequence and real MOT
   metrics. Track counts from an unlabelled video are not a substitute.

> A falling SSL loss is not evidence that detection improved, and a change in `unique_tracks`
> is not evidence that tracking improved.

## 11. Export the results

In [ ]:
summary_path = OUTPUT_ROOT / "tracker_comparison.csv"
results.to_csv(summary_path)

manifest = {
    "lab": "multi-object tracking and data association",
    "ssldet_version": INSTALLED,
    "model_name": MODEL_NAME,
    "weights": str(WEIGHTS),
    "video": str(VIDEO),
    "max_frames": MAX_FRAMES,
    "trackers_compared": list(available_tracker_names()),
    "comparison_csv": str(summary_path),
    "analysis_type": "unlabelled_video",
    "accuracy_metrics_available": False,
    "why": (
        "A plain video has no ground-truth identities, so MOTA/MOTP/IDF1/HOTA and ID-switch "
        "counts are not computable. Only counts, lifetimes, latency, and throughput are."
    ),
}
manifest_path = OUTPUT_ROOT / "tracker_lab_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"comparison : {summary_path}")
print(f"manifest   : {manifest_path}")
print(f"annotated  : {botsort_result.annotated_media}")
display(Markdown(Path(botsort_result.outcome_markdown).read_text(encoding="utf-8")))

## Summary

You ran one detector through six association algorithms and measured what changed.

**What you can state from this notebook**

- Throughput of each tracker on this clip and this hardware.
- Relative track counts and lifetimes on identical detections.
- The exact tracker configuration behind every number, recovered from the report.

**What still requires labels**

- Any claim containing the words *accurate*, *correct*, *better*, or *worse*.
- MOTA, MOTP, IDF1, HOTA, ID switches, fragmentation rate.

### Next

- [Evaluation](https://github.com/rifat963/ssl-detection-lab/blob/main/wiki/Evaluation.md) —
  measure detection accuracy where ground truth exists
- [Video Analysis](https://github.com/rifat963/ssl-detection-lab/blob/main/wiki/Video-Analysis.md) —
  the full report schema and tracker reference
- Re-run this notebook with an SSL-pretrained `best.pt` and a COCO-pretrained one, holding the
  tracker fixed, to see how much of the tracking behaviour is really the detector